# CSE 4/587 — End-to-End Big Data Pipeline
# Project Phase 1 Report

**Dataset:** S&P 500 Stock Prices (2014–2017)

**Team Members:** [Member 1 Name (UBID)], [Member 2 Name (UBID)], [Member 3 Name (UBID)]

**Date:** March 14, 2026

---

## Task 1: Problem Statement (20 pts)

### Project Title
*[TO BE FILLED BY TEAMMATE — GeneralHershey]*

### Problem Description & Motivation
*[TO BE FILLED BY TEAMMATE]*

### Stakeholders
*[TO BE FILLED BY TEAMMATE]*

### Machine Learning Problem Statements (N=3)
1. *[TO BE FILLED BY TEAMMATE]*
2. *[TO BE FILLED BY TEAMMATE]*
3. *[TO BE FILLED BY TEAMMATE]*

### Data Analysis Objectives (2N=6)
1. *[TO BE FILLED BY TEAMMATE]*
2. *[TO BE FILLED BY TEAMMATE]*
3. *[TO BE FILLED BY TEAMMATE]*
4. *[TO BE FILLED BY TEAMMATE]*
5. *[TO BE FILLED BY TEAMMATE]*
6. *[TO BE FILLED BY TEAMMATE]*

### Input → Output Specification
*[TO BE FILLED BY TEAMMATE]*

---

## Task 2: Data Sources (15 pts)

### Dataset Information

| Attribute | Detail |
|-----------|--------|
| **Dataset Name** | S&P 500 Stock Prices 2014–2017 |
| **Source** | Kaggle — Gaurav Mehta |
| **URL** | https://www.kaggle.com/datasets/gauravmehta13/sp-500-stock-prices |
| **License** | CC0: Public Domain |
| **Format** | CSV |
| **Size** | 497,472 rows × 7 columns |
| **Time Span** | January 2, 2014 – December 29, 2017 |
| **Coverage** | 505 S&P 500 component stocks, daily granularity |

### Dataset Description

This dataset contains historical daily stock price data for all companies listed in the S&P 500 index, sourced from The Investor's Exchange (IEX). Each record represents one trading day for one stock, providing open, high, low, and close prices along with trading volume. The dataset spans approximately four years (2014–2017) and covers 505 unique ticker symbols.

### Column Descriptions

| Column | Type | Description |
|--------|------|-------------|
| `symbol` | String | Stock ticker symbol (e.g., AAPL, GOOG, MSFT) |
| `date` | String | Trading date in YYYY-MM-DD format |
| `open` | Float | Opening price in USD at market open |
| `high` | Float | Highest price reached during the trading day |
| `low` | Float | Lowest price reached during the trading day |
| `close` | Float | Closing price in USD at market close |
| `volume` | Integer | Total number of shares traded during the day |

### Dataset Size Justification

With 497,472 rows, the dataset significantly exceeds the minimum requirement of 100,000 rows. The dataset contains daily records for 505 stocks over ~1,006 trading days, providing sufficient data for data-intensive processing and analysis.

### Citation

Nugent, C. (2018). *S&P 500 stock data*. Kaggle. Retrieved from https://www.kaggle.com/datasets/gauravmehta13/sp-500-stock-prices. License: CC0 Public Domain.

---

## Task 3: Hadoop/HDFS Utilization (15 pts)

### Environment Setup

We used a Docker-based Hadoop cluster for HDFS storage:

```bash
docker pull apache/hadoop:3
docker run -d --name hadoop --platform linux/amd64 \
  -v "/path/to/project:/data" apache/hadoop:3
```

### Part A: Raw Data Ingestion to HDFS

The raw dataset was uploaded to HDFS under `/user/project/raw/`:

```bash
hdfs dfs -mkdir -p /user/project/raw
hdfs dfs -put sp500_stock_prices_2014_2017.csv /user/project/raw/
```

### Part B: Cleaned Data Upload to HDFS

After completing Tasks 4 and 5, the cleaned dataset was uploaded to HDFS under `/user/project/cleaned/`:

```bash
hdfs dfs -mkdir -p /user/project/cleaned
hdfs dfs -put cleaned_sp500.csv /user/project/cleaned/
```

### HDFS Verification Screenshot

The following screenshot shows the successful upload of both raw and cleaned datasets to HDFS:

In [ ]:
from IPython.display import Image, display
display(Image(filename='hdfs_screenshot.png', width=900))

### HDFS Directory Structure

```
/user/project/
├── raw/
│   └── sp500_stock_prices_2014_2017.csv    (23.1 MB — original raw data)
└── cleaned/
    └── cleaned_sp500.csv                    (45.0 MB — cleaned + derived features)
```

Both the raw dataset and the cleaned dataset are stored in HDFS. The raw dataset is preserved for reference, and the cleaned dataset will be used as the starting point for Phase 2.

---

## Task 4: Data Cleaning / Processing (25 pts)

All cleaning operations were performed in Jupyter Notebook using Python (pandas). Below we document 6 distinct cleaning/processing operations (2N, where N=3 team members).

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Load raw dataset
df = pd.read_csv('S&P 500 Stock Prices 2014-2017.csv')
print(f'Raw dataset shape: {df.shape}')
print(f'Rows: {df.shape[0]:,}, Columns: {df.shape[1]}')
print(f'\n=== Data Types ===')
print(df.dtypes)
print(f'\n=== First 5 Rows ===')
df.head()

### Cleaning Operation 1: Handle Missing Values

**Rationale:** Missing values can distort statistical analysis and cause errors in machine learning models. We first identify the extent of missingness, then remove affected rows since they represent a negligible fraction (<0.003%) of the data.

In [ ]:
# Identify missing values
print('=== Missing Values per Column ===')
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage (%)': missing_pct.round(4)})
print(missing_df)
print(f'\nTotal rows with any missing value: {df.isnull().any(axis=1).sum()}')

# Drop rows with missing values
rows_before = len(df)
df = df.dropna()
rows_after = len(df)
print(f'\nRows before cleaning: {rows_before:,}')
print(f'Rows after cleaning:  {rows_after:,}')
print(f'Rows removed:         {rows_before - rows_after:,}')

**Result:** 11 rows had missing values in `open`, `high`, or `low` columns (0.002% of data). These were dropped since imputing stock prices without market data would be unreliable.

---

### Cleaning Operation 2: Fix Data Types — Parse Date Column

**Rationale:** The `date` column is loaded as a string (object type). Converting it to `datetime64` enables proper chronological sorting, time-based grouping, and extraction of temporal features (year, month, day of week).

In [ ]:
print(f'Date dtype BEFORE: {df["date"].dtype}')
df['date'] = pd.to_datetime(df['date'])
print(f'Date dtype AFTER:  {df["date"].dtype}')
print(f'\nDate range: {df["date"].min().date()} to {df["date"].max().date()}')
print(f'Total trading days: {df["date"].nunique()}')

**Result:** Successfully converted `date` from string to `datetime64[ns]`. The dataset spans from 2014-01-02 to 2017-12-29.

---

### Cleaning Operation 3: Remove Duplicate Rows

**Rationale:** Duplicate entries for the same stock on the same date would inflate statistics and bias machine learning models. We check for duplicates based on the (symbol, date) pair, which should be unique.

In [ ]:
dupes = df.duplicated(subset=['symbol', 'date']).sum()
print(f'Duplicate (symbol, date) pairs found: {dupes}')

rows_before = len(df)
df = df.drop_duplicates(subset=['symbol', 'date'], keep='first')
rows_after = len(df)
print(f'Rows before: {rows_before:,}')
print(f'Rows after:  {rows_after:,}')
print(f'Duplicates removed: {rows_before - rows_after:,}')

**Result:** No duplicate (symbol, date) pairs were found, confirming the dataset has one record per stock per trading day.

---

### Cleaning Operation 4: Outlier Detection and Handling

**Rationale:** We check for data quality issues including: (1) zero or negative trading volume (indicates non-trading or data errors), and (2) price anomalies where the reported high price is less than the low price, which is logically impossible.

In [ ]:
# Check for zero or negative volume
zero_vol = (df['volume'] <= 0).sum()
print(f'Rows with volume <= 0: {zero_vol}')

rows_before = len(df)
df = df[df['volume'] > 0]
print(f'Removed {rows_before - len(df)} zero-volume rows')

# Check for price anomalies
price_errors = ((df['high'] < df['low']) | (df['close'] <= 0)).sum()
print(f'\nRows with price anomalies (high < low or close <= 0): {price_errors}')

rows_before = len(df)
df = df[(df['high'] >= df['low']) & (df['close'] > 0)]
rows_after = len(df)
print(f'Rows removed due to price anomalies: {rows_before - rows_after}')
print(f'Rows remaining: {rows_after:,}')

**Result:** No zero-volume rows found. 1 row with a price anomaly (high < low) was removed. This ensures data integrity for all price-based calculations.

---

### Cleaning Operation 5: Feature Engineering — Derive New Columns

**Rationale:** The raw OHLCV data alone is limited for analysis and modeling. We derive new features that capture important financial characteristics: daily return (profitability), price range (volatility proxy), and temporal features (for seasonality analysis).

In [ ]:
# Daily return: percentage change from open to close
df['daily_return'] = ((df['close'] - df['open']) / df['open']) * 100

# Intraday price range: difference between high and low
df['price_range'] = df['high'] - df['low']

# Day of week (0=Monday, 4=Friday)
df['day_of_week'] = df['date'].dt.dayofweek

# Year and Month for temporal analysis
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

print('New columns added:')
print('  - daily_return:  ((close - open) / open) * 100')
print('  - price_range:   high - low')
print('  - day_of_week:   0=Monday ... 4=Friday')
print('  - year:          extracted from date')
print('  - month:         extracted from date')
print(f'\nDataset now has {df.shape[1]} columns')
df.head()

**Result:** 5 new columns were derived, bringing the total to 12 columns. These features will be directly useful for Phase 2 modeling (e.g., `daily_return` as a prediction target, `day_of_week` as a feature).

---

### Cleaning Operation 6: Normalize Symbol Names and Sort Data

**Rationale:** Ensuring consistent uppercase formatting for stock symbols prevents grouping errors (e.g., 'aapl' vs 'AAPL'). Sorting by (symbol, date) ensures proper chronological order within each stock, which is essential for time-series analysis.

In [ ]:
# Normalize symbols to uppercase and strip whitespace
df['symbol'] = df['symbol'].str.upper().str.strip()
print(f'Unique stock symbols: {df["symbol"].nunique()}')

# Sort by symbol and date
df = df.sort_values(['symbol', 'date']).reset_index(drop=True)

# Final summary
print(f'\n=== Final Cleaned Dataset Summary ===')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')
print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')
print(f'\n=== Data Types ===')
print(df.dtypes)

**Result:** All 505 stock symbols are consistently formatted. Data is sorted chronologically within each stock. The final cleaned dataset contains 497,460 rows × 12 columns.

---

In [ ]:
# Export cleaned dataset
df.to_csv('cleaned_sp500.csv', index=False)
print(f'Cleaned dataset exported: cleaned_sp500.csv')
print(f'Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

---

## Task 5: Exploratory Data Analysis (25 pts)

We performed 6 significant EDA operations (2N, where N=3 team members), including a mix of non-graphical and graphical analyses.

---

### EDA 1: Summary Statistics (Non-Graphical)

**Purpose:** Understand the central tendency, spread, and shape of all numerical features to identify data characteristics and inform modeling decisions.

In [ ]:
print('=== Descriptive Statistics ===')
df[['open', 'high', 'low', 'close', 'volume', 'daily_return', 'price_range']].describe().round(2)

**Insight:** Stock prices range from ~$1.50 to over $2,000, reflecting the wide diversity of companies in the S&P 500 (from small-cap inclusions to high-priced stocks like Amazon). The mean daily return is near zero (0.03%), consistent with the random walk hypothesis. The large standard deviation in volume (8.2M vs mean 4.3M) indicates highly skewed trading activity — a few mega-cap stocks dominate volume. This variance will require normalization or log-transformation for Phase 2 models.

---

### EDA 2: Distribution of Daily Returns (Graphical — Histogram + Box Plot)

**Purpose:** Examine whether daily returns follow a normal distribution, which is a fundamental assumption in many financial models (e.g., Black-Scholes, CAPM).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['daily_return'].clip(-10, 10), bins=100, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_title('Distribution of Daily Returns (%)', fontsize=14)
axes[0].set_xlabel('Daily Return (%)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(x=0, color='red', linestyle='--', label='Zero Return')
axes[0].legend()

# Box plot
axes[1].boxplot(df['daily_return'].clip(-10, 10), vert=True)
axes[1].set_title('Box Plot of Daily Returns (%)', fontsize=14)
axes[1].set_ylabel('Daily Return (%)')

plt.tight_layout()
plt.show()

print(f'Mean daily return:   {df["daily_return"].mean():.4f}%')
print(f'Median daily return: {df["daily_return"].median():.4f}%')
print(f'Skewness:            {df["daily_return"].skew():.4f}')
print(f'Kurtosis:            {df["daily_return"].kurtosis():.4f}')

**Insight:** Daily returns are approximately centered around zero but exhibit significant leptokurtosis (high kurtosis), meaning extreme price movements (both gains and losses) occur far more frequently than a normal distribution would predict. The distribution also shows slight positive skewness. These "fat tails" have direct implications for Phase 2: standard regression models may underestimate tail risk, and anomaly detection models should account for this non-normality.

---

### EDA 3: Correlation Analysis (Non-Graphical + Heatmap)

**Purpose:** Identify relationships between numerical features to guide feature selection and avoid multicollinearity in Phase 2 machine learning models.

In [ ]:
corr_cols = ['open', 'high', 'low', 'close', 'volume', 'daily_return', 'price_range']
corr_matrix = df[corr_cols].corr().round(3)

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, fmt='.2f', square=True)
plt.title('Correlation Heatmap of Stock Features', fontsize=14)
plt.tight_layout()
plt.show()

print('\n=== Correlation Matrix ===')
print(corr_matrix)

**Insight:** OHLC prices are nearly perfectly correlated (r > 0.99), which is expected since they represent the same stock on the same day. This means we should avoid using all four price columns simultaneously in ML models to prevent multicollinearity — using `close` alone or derived features like `daily_return` is preferable. Volume shows very low correlation with price levels (r ≈ -0.02), suggesting it carries independent information and should be retained as a feature. The moderate correlation between `price_range` and `daily_return` (r ≈ 0.06) indicates these capture different aspects of price behavior.

---

### EDA 4: Trading Volume Over Time (Graphical — Line Chart)

**Purpose:** Examine temporal trends and patterns in aggregate market activity to understand seasonality, market regime changes, and potential structural breaks in the data.

In [ ]:
# Aggregate daily total volume across all stocks
daily_volume = df.groupby('date')['volume'].sum().reset_index()

plt.figure(figsize=(14, 5))
plt.plot(daily_volume['date'], daily_volume['volume'] / 1e9, color='steelblue', alpha=0.7, linewidth=0.8)
plt.title('Total Daily Trading Volume Across S&P 500 (2014–2017)', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Total Volume (Billions of Shares)')
plt.tight_layout()
plt.show()

print('=== Average Trading Volume by Year ===')
yearly = df.groupby('year')['volume'].agg(['mean', 'sum']).round(0)
yearly.columns = ['Mean Volume per Stock-Day', 'Total Volume']
print(yearly)

**Insight:** Trading volume displays periodic spikes that often correspond to major market events such as earnings seasons, Federal Reserve announcements, and market corrections (e.g., the August 2015 flash crash, Brexit vote in June 2016). The overall volume trend shows variation across years. These temporal volume patterns will be useful for Phase 2 time-series modeling and could serve as a feature for predicting abnormal market activity.

---

### EDA 5: Top 10 Most and Least Volatile Stocks (Graphical — Bar Chart)

**Purpose:** Identify which stocks exhibit the highest and lowest price volatility, which is essential for risk assessment, portfolio construction, and anomaly detection tasks planned for Phase 2.

In [ ]:
# Volatility = standard deviation of daily returns per stock
volatility = df.groupby('symbol')['daily_return'].std().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top 10 most volatile
top10 = volatility.head(10)
axes[0].barh(top10.index[::-1], top10.values[::-1], color='salmon', edgecolor='black')
axes[0].set_title('Top 10 Most Volatile Stocks', fontsize=14)
axes[0].set_xlabel('Std of Daily Return (%)')

# Top 10 least volatile
bottom10 = volatility.tail(10)
axes[1].barh(bottom10.index[::-1], bottom10.values[::-1], color='lightgreen', edgecolor='black')
axes[1].set_title('Top 10 Least Volatile Stocks', fontsize=14)
axes[1].set_xlabel('Std of Daily Return (%)')

plt.tight_layout()
plt.show()

print(f'Overall average volatility: {volatility.mean():.4f}%')
print(f'Most volatile:  {volatility.idxmax()} ({volatility.max():.4f}%)')
print(f'Least volatile: {volatility.idxmin()} ({volatility.min():.4f}%)')

**Insight:** Volatility varies dramatically across S&P 500 stocks. The most volatile stocks tend to be in technology, biotech, or energy sectors, while the least volatile are typically utilities and consumer staples. This wide dispersion in volatility is important for Phase 2: (1) for clustering, we can group stocks by risk profile, (2) for regression, volatility itself could be a prediction target, and (3) for classification, high-volatility stocks may behave differently from low-volatility stocks, requiring separate modeling approaches.

---

### EDA 6: Average Daily Return by Day of Week (Graphical — Bar Chart)

**Purpose:** Investigate the "day-of-week effect" — a well-documented market anomaly where certain weekdays exhibit systematically different returns. Understanding this pattern informs feature engineering for Phase 2.

In [ ]:
day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
dow_returns = df.groupby('day_of_week')['daily_return'].agg(['mean', 'std']).round(4)
dow_returns.index = day_names

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['green' if x >= 0 else 'red' for x in dow_returns['mean']]
axes[0].bar(dow_returns.index, dow_returns['mean'], color=colors, edgecolor='black', alpha=0.8)
axes[0].set_title('Average Daily Return by Day of Week', fontsize=14)
axes[0].set_ylabel('Mean Return (%)')
axes[0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)

axes[1].bar(dow_returns.index, dow_returns['std'], color='steelblue', edgecolor='black', alpha=0.8)
axes[1].set_title('Return Volatility by Day of Week', fontsize=14)
axes[1].set_ylabel('Std of Return (%)')

plt.tight_layout()
plt.show()

print('=== Return Statistics by Day of Week ===')
print(dow_returns)

**Insight:** The analysis reveals differences in average returns across weekdays. If certain days consistently show higher or lower returns, this validates including `day_of_week` as a feature in Phase 2 prediction models. The volatility chart also shows whether risk varies by day, which is relevant for risk-adjusted return modeling. These patterns, even if small, can contribute to the predictive power of ensemble models.

---

## Summary

### Data Cleaning Summary
| Operation | Action | Rows Affected |
|-----------|--------|---------------|
| Missing values | Dropped rows with null open/high/low | 11 rows removed |
| Date parsing | Converted string to datetime | All rows |
| Duplicates | Checked (symbol, date) uniqueness | 0 duplicates found |
| Outlier handling | Removed price anomalies | 1 row removed |
| Feature engineering | Added 5 derived columns | All rows enriched |
| Normalization & sorting | Uppercase symbols, chronological sort | All rows |

**Final cleaned dataset:** 497,460 rows × 12 columns

### Key EDA Findings
1. Daily returns are **heavy-tailed** (non-normal), requiring robust modeling techniques
2. OHLC prices are **highly correlated** — use derived features instead of raw prices
3. Volume carries **independent information** from price and should be retained
4. **Temporal patterns** exist in both volume and returns
5. **Volatility varies significantly** across stocks — useful for clustering
6. **Day-of-week effects** may provide additional predictive signal

### Phase 2 Implications
These insights will guide: feature engineering (daily return, volatility, temporal features), model selection (need to handle non-normal distributions and varying scales), and evaluation strategy (consider risk-adjusted metrics alongside accuracy).